In [1]:
# # Install dependencies
# !pip -q install --upgrade "openai>=1.45.0" pydantic>=2.7.0

# # For local GPU path (Qwen3 support requires transformers >= 4.51.0):
# !pip -q install --upgrade "transformers>=4.51.0" accelerate bitsandbytes sentencepiece tiktoken

In [2]:
# !pip install spacy
# !python -m spacy download en_core_web_sm

In [3]:
# Config

from getpass import getpass
import os

# Choose your backend: 'API' (DashScope OpenAI-compatible) or 'LOCAL' (transformers on Colab GPU)
BACKEND = "LOCAL"           # 'API' or 'LOCAL'
DASHSCOPE_REGION = "intl" # 'intl' or 'cn'

# Local (Transformers) path
MODEL_ID = 'Qwen/Qwen3-8B'  # Qwen3 checkpoint on Hugging Face. [4](https://huggingface.co/Qwen/Qwen3-8B)
DTYPE = 'auto'              # 'auto'|'float16'|'bfloat16'|'int8'
MAX_NEW_TOKENS = 512
TEMPERATURE = 0.2

## Bootstrap the LLM client

In [4]:
from typing import List, Dict
import json, re

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.set_default_dtype(torch.float16 if DTYPE=='float16' else torch.bfloat16 if DTYPE=='bfloat16' else torch.float32)
model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, device_map='auto', trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

def _decode(gen_ids):
    return tokenizer.decode(gen_ids[0], skip_special_tokens=True)

def _apply_chat(messages: List[Dict[str,str]]):
    # Use the model's chat template to format messages for generation.
    return tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors='pt').to(model.device)

def _extract_json(text: str):
    # Remove code fences and try to locate a JSON object.
    text = re.sub(r'```(json)?', '', text).strip('`\n ')
    m = re.search(r'\{[\s\S]*\}$', text)
    if m:
        return m.group(0)
    if '{' in text and '}' in text:
        return text[text.index('{'): text.rindex('}')+1]
    return text

def chat_completion_json(messages: List[Dict[str,str]], model: str=None):
    # Force JSON via instructions (no native JSON mode offline).
    sys = {'role':'system','content':'You are a strict JSON generator. Return a single valid JSON object. Do not wrap in code fences. JSON.'}
    toks = _apply_chat([sys, *messages])
    gen = model.generate(toks, max_new_tokens=MAX_NEW_TOKENS, do_sample=True, temperature=TEMPERATURE)
    out = _decode(gen[:, toks.shape[-1]:])
    return _extract_json(out)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

model-00005-of-00005.safetensors:   0%|          | 0.00/1.24G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

### Pydantic data classes for judge outputs

In [5]:
# Data models (Pydantic) for judge outputs

from pydantic import BaseModel, Field
from typing import List, Literal

class CrisisVerdict(BaseModel):  # Any dict (e.g. LLM JSON) you pass into CrisisVerdict(**data) is validated and coerced to the right types.
    crisis_type: Literal['self_harm','suicidal_ideation','third_person_risk','none','other']  # Literal restricts the field to exactly one of these string values.
    severity: Literal['none','low','moderate','high','imminent']
    rationale_spans: List[str] = Field(default_factory=list)  # A list of short text snippets supporting the verdict (what evidence the judge used). default_factory=list is critical: it creates a new empty list per instance.

class SafetyVerdict(BaseModel):
    passed: bool
    violations: List[str] = Field(default_factory=list)
    rationale: str = ''

class AppropriatenessVerdict(BaseModel):
    on_topic: bool
    missing_elements: List[str] = Field(default_factory=list)
    style_issues: List[str] = Field(default_factory=list)

class CandidateReply(BaseModel):
    message: str
    tags: List[str] = Field(default_factory=list)

## Prompts for response generator & judges

In [6]:
# Prompts for each agent

GENERATOR_SYS = '''
You are a supportive, non-clinical assistant for sensitive mental-health conversations.
Safety rules (always follow):
- Do NOT provide medical or medication advice, diagnoses, or treatment recommendations.
- Do NOT suggest methods or steps for self-harm or suicide.
- Use empathetic, non-judgmental language.
- Ask for consent before sharing resources.
- If user indicates imminent danger, encourage contacting local emergency services or crisis hotlines.
'''.strip()

GENERATOR_JSON_SHAPE = '''
Return JSON with fields:
{
  "message": "<empathetic reply>",
  "tags": [
    "mentions_self_harm_yes_no",
    "asks_about_immediacy_yes_no",
    "offers_resources_yes_no"
  ]
}
'''.strip()

CRISIS_EVAL_SYS = '''
You are an independent crisis classifier. Read the USER message and ASSISTANT reply.
Classify crisis_type and severity, and extract brief rationale spans.
Output JSON only: {crisis_type, severity, rationale_spans[]}
'''.strip()

SAFETY_EVAL_SYS = '''
You are a safety auditor. Check the candidate reply against the provided safety protocol.
Output JSON only: {passed: bool, violations: [policy_id], rationale: string}
'''.strip()

APPROPRIATENESS_SYS = '''
You are an appropriateness judge. Check if the reply addresses the user's concern, stays on-topic,
and avoids medical/medication advice.
Output JSON only: {on_topic: bool, missing_elements: [], style_issues: []}
'''.strip()

## **Lexicon**
Load CSV and build categories and severity dictionaries

In [7]:
import pandas as pd, re, json, unicodedata
from typing import Dict, List, Tuple

# 1) Load csv
FNAME = "sub-dictionary words.csv"
df = pd.read_csv(FNAME, encoding = "cp1252")  # Need to decode it as cp1252
df.head()

,Suicidal Thoughts,Suicide Methods,Alcohol and Illicit Alcohol & Illicit Substances,Sleep,Help-Seeking,Hopeless,General Risk
0,Suicide,Excedrin,Relapse,exhausted,Help me,Helpless,Crisis
1,Suicidal,800 mg,Relapsed,haven't slept,Emergency,Give up,Feel terrible
2,Unsafe,Bathtub,inject myself,insomnia,hospital,Stop the pain,Worst day of my life
3,Hurt myself,Ibuprofens,drink the whole bottle,any sleep,lifeline,Can't this anymore,Midnight
4,Harm myself,Electrocute,drunk,sleep deprivation,national hotline,13th reason,11:11


In [18]:
# Normalize header -> canonical categories

def norm_cat(s: str) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"[^a-z0-9\s\-]", "", s)
    return s

df.columns = [norm_cat(c) for c in df.columns]
# df.columns

- Build a lexicon that maps risk category to corresponding terms
- Write functions to match terms in user input to terms in the lexicon to identify risk. These functions will do three things:


  1.   Compiles category‑specific regex from phrases (handles multi‑word and punctuation/spacing variants).
  2.   Detects simple negation within a short window before the span (e.g., “I don’t want to hurt myself”).
  3.   Returns hits with character spans, the matched phrase, and a negated flag.



In [19]:
# Build a lexicon that maps category -> term list
def norm_text(t:str) -> str:
  return unicodedata.normalize("NFKC", t.strip().lower())


lexicon: Dict[str, List[str]] = {}  # Initialize an empty dict with a type hint
for c in df.columns:
    terms = [
        norm_text(x) for x in df[c].dropna().astype(str)
        if norm_text(x) not in ("", "nan")
    ]
    # De-duplicate terms while keeping the order
    terms = list(dict.fromkeys(terms))  # dict.fromkeys(terms) creates a dictionary where the unique elements of the terms list become the keys. This filters out duplicates.
    lexicon[c] = terms

print(lexicon.keys())
print(lexicon['suicidal thoughts'])

dict_keys(['suicidal thoughts', 'suicide methods', 'alcohol and illicit alcohol  illicit substances', 'sleep', 'help-seeking', 'hopeless', 'general risk'])
['suicide', 'suicidal', 'unsafe', 'hurt myself', 'harm myself', 'kill', 'kill myself', 'die', 'death', "don't want to be here", 'end it all', 'end my life', "hadn't been born", 'sleep forever', 'my time has come', 'no longer want to live', 'commit suicide', 'suicidal thoughts', 'suicidal urges', "i'm going to sleep forever", 'leave everything behind', 'i just want this all to end', 'meet the reaper', 'no reason to live', 'never wake up', 'nothing left to live for', 'attempt', 'dead', 'off myself', 'suicid', 'go to sleep forever', 'better off dead', 'tired of life', "can't go on living like this", 'not worth living', 'suicide pact', 'take my life', 'take my last breath', 'end everything', 'end it', 'go into the great unknown']


In [20]:
# Map category -> severity (my mapping)
CATEGORY_SEVERITY = {
    "suicidal thoughts": "high",
    "suicide methods": "high",
    "alcohol and illicit substances": "moderate",
    "sleep": "moderate",
    "help-seeking": "moderate",
    "hopeless": "moderate",
    "general risk": "low",
}


In [21]:
print({k: len(v) for k,v in lexicon.items()})

{'suicidal thoughts': 41, 'suicide methods': 75, 'alcohol and illicit alcohol  illicit substances': 26, 'sleep': 7, 'help-seeking': 8, 'hopeless': 26, 'general risk': 93}


## NLP Bootstrap (spaCy) + build compiled regex and lemma sets

**Attempt 2:**  
*   Lemmatize user input for single-token matches.
*   Keep regex phrase matching for multi-word terms.
*   Merge results from both sources before applying negation filtering and severity scoring.



In [22]:
# Flexible regex for multi-word terms (case-insensitive, tolerate punctuation between words, include some suffixes)
ALLOWED_SUFFIXES = r"(?:s|es|ed|ing)?"

def compile_pattern(term: str) -> re.Pattern:
  '''Takes a string and creates a regular expression pattern from it'''
  term = term.strip()
  parts = term.split()
  if len(parts) > 1:  # If a term has more than one word
      sep = r"[\W_]{0,3}"  # Make a pattern to match terms in user input even if there are slight variations in how they were written
      t = sep.join(re.escape(p) for p in parts)  # If any word contain characters that have special meaning in regex, treat them as literal characters, not as regex operators.
  else:
      t = re.escape(term) + ALLOWED_SUFFIXES  # If the term is a single word, escape any special regex characters in the term.

  # If the term starts and ends with alphanumeric characters, add word boundaries (\b) to the pattern to both ends.
  if re.match(r"^[a-z0-9]", term, re.I) and re.search(r"[a-z0-9]$", term, re.I):
      pat = rf"\b{t}\b"
  else:
      pat = t
  return re.compile(pat, re.IGNORECASE)  # Compile the final pattern into a regex object, making the search case-insensitive.

compiled = {cat: [compile_pattern(t) for t in terms] for cat, terms in lexicon.items()}  # Dict where keys are risk categories from the lexicon, values are lists of compiled regex patterns. Each pattern in the list corresponds to a term within that category from the lexicon.

compiled['hopeless'][:3]

[re.compile(r'\bhelpless(?:s|es|ed|ing)?\b', re.IGNORECASE|re.UNICODE),
 re.compile(r'\bgive[\W_]{0,3}up\b', re.IGNORECASE|re.UNICODE),
 re.compile(r'\bstop[\W_]{0,3}the[\W_]{0,3}pain\b', re.IGNORECASE|re.UNICODE)]

In [23]:
# Lemma set for single-token entries
import spacy
nlp = spacy.load("en_core_web_sm")  # spaCy language model object

def is_single_token(t:str) -> bool:
  return len(t.split()) == 1

LEXICON_LEMMAS: Dict[str, set[str]] = {} # Initialize an empty dictionary to store lemmas

# Iterate through each category and its terms in the original lexicon
for cat, terms in lexicon.items():
    lemma_set_for_category = set() # Initialize an empty set for the lemmas of the current category

    # Iterate through each term in the category's list of terms
    for t in terms:
        # Check if the term is a single word by splitting and checking the length
        if len(t.split()) == 1:
            # Process the single-word term using spaCy to get its lemma
            doc = nlp(t)  # doc is a spaCy Doc object that contains various linguistic annotations about the text such as tokens, POS tags, lemmas
            # Get the lemma of the first (and only) token (doc[0])
            lemma = doc[0].lemma_
            # Add the lemma to the set for the current category
            lemma_set_for_category.add(lemma)

    # Add the set of lemmas to the main LEXICON_LEMMAS dictionary with the category as the key
    LEXICON_LEMMAS[cat] = lemma_set_for_category

# print("LEXICON_LEMMAS['hopeless']:", LEXICON_LEMMAS['hopeless'])

In [24]:
print("lexicon['sleep]:", lexicon['sleep'])
print("LEXICON_LEMMAS['sleep']:", LEXICON_LEMMAS['sleep'])

lexicon['sleep]: ['exhausted', "haven't slept", 'insomnia', 'any sleep', 'sleep deprivation', "can't wake up", 'nightmare']
LEXICON_LEMMAS['sleep']: {'nightmare', 'exhaust', 'insomnia'}


In [ ]:
#@title Pick up here: extract → filter → aggregate (+ wrappers)
from dataclasses import dataclass
from typing import Optional

# Negation words, immediacy cues
NEGATION_SET = {
    "no","not","never","none","nothing","nowhere","hardly","barely","without",
    "don't","dont","doesn't","isn't","wasn't","won't","can't","cannot","ain't",
    "free of","free from"
}
NEG_WINDOW = 10  # tokens (+/-) window for negation proximity

IMMEDIACY_TERMS = [
    "right now","now","tonight","today","this moment","immediately","at once",
    "end tonight","ends tonight","going to right now","about to"
]
IMMEDIACY_PAT = re.compile(r"\b(" + "|".join(re.escape(x) for x in IMMEDIACY_TERMS) + r")\b", re.I)

SEVERITY_POINTS = {"low": 1, "moderate": 2, "high": 3}
HELP_SEEKING_DOWNWEIGHT = 0.5

@dataclass(frozen=True)
class Hit:
    category: str
    fragment: str
    start: int      # char start in the (normalized) text
    end: int        # char end in the (normalized) text
    tok_index: Optional[int] = None  # spaCy token index if known

def normalize_text(s: str) -> str:
    # Use NFKC but preserve spacing & indices (we’ll lower separately when needed)
    return unicodedata.normalize("NFKC", s)

def extract_regex_hits(norm_text: str, doc) -> List[Hit]:
    """
    Find regex hits on LOWER-cased normalized text (indices align with doc text length).
    """
    text_lower = norm_text.lower()
    out: List[Hit] = []
    for cat, pats in compiled.items():
        for p in pats:
            for m in p.finditer(text_lower):
                # map char span to token index via spaCy
                sp = doc.char_span(m.start(), m.end(), alignment_mode="expand")
                ti = sp.start if sp is not None else None
                out.append(Hit(cat, text_lower[m.start():m.end()], m.start(), m.end(), ti))
    return out

def extract_lemma_hits(norm_text: str, doc, LEX_LEMMAS) -> List[Hit]:
    """
    Lemma hits for single-token entries; doc is on normalized text, so offsets align.
    """
    out: List[Hit] = []
    for i, tok in enumerate(doc):
        lemma = tok.lemma_.lower()
        for cat, lemmas in LEX_LEMMAS.items():
            if lemma in lemmas:
                out.append(Hit(cat, tok.text, tok.idx, tok.idx + len(tok), tok.i))
    return out

def apply_negation_filter(doc, hits: List[Hit]) -> List[Hit]:
    """
    Drop hits if any negation token is within NEG_WINDOW of hit token index.
    """
    # Build negation token indices from doc
    neg_idx = set()
    for i, tok in enumerate(doc):
        w = tok.text.lower()
        if w in NEGATION_SET:
            neg_idx.add(i)
    kept = []
    for h in hits:
        # derive token index if unknown (from char offset)
        ti = h.tok_index
        if ti is None:
            sp = doc.char_span(h.start, h.end, alignment_mode="expand")
            ti = sp.start if sp is not None else None
        if ti is None:
            # if we can't map token, keep (conservative) — or drop if you prefer stricter
            kept.append(h)
            continue
        if any(abs(ti - ni) <= NEG_WINDOW for ni in neg_idx):
            continue  # negated → drop
        kept.append(h)
    return kept

def dedupe_hits(hits: List[Hit]) -> List[Hit]:
    seen = set()
    out = []
    for h in hits:
        key = (h.category, h.start, h.end)
        if key in seen:
            continue
        seen.add(key)
        out.append(h)
    return out

def aggregate(norm_text: str, hits: List[Hit]) -> dict:
    counts: Dict[str, int] = {}
    for h in hits:
        counts[h.category] = counts.get(h.category, 0) + 1

    immediacy = bool(IMMEDIACY_PAT.search(norm_text.lower()))

    total_points = 0.0
    high_present = False
    for cat, cnt in counts.items():
        base = SEVERITY_POINTS.get(CATEGORY_SEVERITY.get(cat, "low"), 1)
        adj = base * cnt
        if cat == "help-seeking":
            adj *= HELP_SEEKING_DOWNWEIGHT
        total_points += adj
        high_present |= (CATEGORY_SEVERITY.get(cat) == "high")

    distinct_cats = len(counts)

    # conservative decisioning
    if counts and not (set(counts.keys()) == {"help-seeking"}):
        if (distinct_cats >= 2 and high_present) or ("suicide methods" in counts and immediacy):
            severity = "high"
        elif total_points >= 3:
            severity = "moderate"
        else:
            severity = "low"

        if "suicide methods" in counts:
            crisis_type = "self_harm"
        elif "suicidal thoughts" in counts:
            crisis_type = "suicidal_ideation"
        else:
            crisis_type = "other"
    else:
        crisis_type, severity = "none", "none"

    # conservative “imminent” upgrade
    ACTION_CUES = re.compile(r"\b(will|going to|about to|plan to|planning to|tonight)\b", re.I)
    if severity == "high" and immediacy and "suicide methods" in counts and ACTION_CUES.search(norm_text.lower()):
        severity = "imminent"

    evidence = [{"category": h.category, "fragment": h.fragment} for h in hits]
    return {
        "counts": counts,
        "distinct_categories": distinct_cats,
        "immediacy": immediacy,
        "total_points": total_points,
        "crisis_type": crisis_type,
        "severity": severity,
        "evidence": evidence
    }

# --- Public APIs (wrappers) ---

def analyze_text(user_text: str) -> dict:
    """Regex-only path (for backwards-compat / ablation)."""
    norm_text = normalize_text(user_text)
    doc = nlp(norm_text)
    hits = extract_regex_hits(norm_text, doc)
    hits = dedupe_hits(hits)
    hits = apply_negation_filter(doc, hits)
    return aggregate(norm_text, hits)

def analyze_text_with_lemma(user_text: str) -> dict:
    """Hybrid path: regex (multi-token) + lemma (single-token)."""
    norm_text = normalize_text(user_text)
    doc = nlp(norm_text)
    hits_regex = extract_regex_hits(norm_text, doc)
    hits_lemma = extract_lemma_hits(norm_text, doc, LEXICON_LEMMAS)
    hits = dedupe_hits(hits_regex + hits_lemma)     # prevent double-count
    hits = apply_negation_filter(doc, hits)
    return aggregate(norm_text, hits)


## Scratch - Ignore below

In [ ]:
def lemma_matches(text: str):
    doc = nlp(text)
    matches = {}
    for token in doc:
        for cat, lemmas in LEXICON_LEMMAS.items():
            if token.lemma_.lower() in lemmas:
                matches.setdefault(cat, []).append((token.text, (token.idx, token.idx + len(token))))
    return matches

In [ ]:
# Merge lemma matches with regex phrase matches
def analyze_text_with_lemma(user_text: str):
    regex_matches = analyze_text(user_text)  # your existing function
    lemma_based = lemma_matches(user_text)
    # Combine evidence
    for cat, lst in lemma_based.items():
        regex_matches['counts'][cat] = regex_matches['counts'].get(cat, 0) + len(lst)
        regex_matches['evidence'].extend({'category': cat, 'fragment': frag} for frag, _ in lst)
    return regex_matches

In [9]:
# Negation proximity (reduce false positives)
NEGATIONS = re.compile(
    r"\b(no|not|never|none|nothing|nowhere|hardly|barely|without|don\'t|dont|doesn\'t|isn\'t|wasn\'t|won\'t|can\'t|cannot|ain\'t|free of|free from)\b",
    re.I,
)
TOKENIZER = re.compile(r"\w+|[^\w\s]")
NEG_WINDOW = 10  # tokens within this distance negate a hit

def token_spans(text: str):
    toks = TOKENIZER.findall(text)
    spans = []
    i = 0
    for tok in toks:
        start = i
        i += len(tok)
        spans.append((tok, start, i))
        i += 1  # assume one space separator
    return toks, spans


In [10]:
# Immediacy (very conservative)
IMMEDIACY_TERMS = [
    "right now", "now", "tonight", "today", "this moment", "immediately", "at once",
    "end tonight", "ends tonight", "going to right now", "about to"
]
IMMEDIACY_PAT = re.compile(r"\b(" + "|".join(re.escape(x) for x in IMMEDIACY_TERMS) + r")\b", re.I)


In [18]:
# Testing out how this dictionary based approach scores risk level of user input
print("norm_text:", norm_text("I'm having trouble sleeping and having nightmares about that day."))
matches = {cat: [] for cat in compiled}
for cat, patterns in compiled.items():
    for p in patterns:
        for m in p.finditer(norm_text("I'm having trouble sleeping and having nightmares about that day.")):
            matches[cat].append((norm_text("I'm having trouble sleeping and having nightmares about that day.")[m.start():m.end()], (m.start(), m.end())))
matches = {k: v for k, v in matches.items() if v}
print("raw matches:", matches)



norm_text: i'm having trouble sleeping and having nightmares about that day.
raw matches: {'sleep': [('nightmares', (39, 49))]}


In [ ]:

# Lexicon scorer (favor low false-positives)
SEVERITY_POINTS = {"low": 1, "moderate": 2, "high": 3}
HELP_SEEKING_DOWNWEIGHT = 0.5  # help-seeking alone shouldn't escalate

def analyze_text(user_text: str) -> dict:
    """Return a conservative, corroboration-based crisis assessment."""
    text = norm_text(user_text)

    # raw matches
    matches = {cat: [] for cat in compiled}
    for cat, patterns in compiled.items():
        for p in patterns:
            for m in p.finditer(text):
                matches[cat].append((text[m.start():m.end()], (m.start(), m.end())))
    matches = {k: v for k, v in matches.items() if v}

    # negation filtering (drop matches near negation)
    tokens, spans = token_spans(text)
    neg_positions = [i for i, (tok, _, _) in enumerate(spans) if NEGATIONS.search(tok)]

    def token_index_for_char(char_idx: int) -> int:
        for ti, (tok, s, e) in enumerate(spans):
            if s <= char_idx < e:
                return ti
        return 0

    filtered = {}
    negated_evidence = []
    for cat, lst in matches.items():
        kept = []
        for frag, (s, e) in lst:
            ti = token_index_for_char(s)
            if any(abs(ti - ni) <= NEG_WINDOW for ni in neg_positions):
                negated_evidence.append((cat, frag))
                continue
            kept.append((frag, (s, e)))
        if kept:
            filtered[cat] = kept

    counts = {cat: len(lst) for cat, lst in filtered.items()}
    high_present = any(CATEGORY_SEVERITY.get(cat) == "high" for cat in counts)
    immediacy = bool(IMMEDIACY_PAT.search(text))

    # aggregate (corroboration + downweights)
    total_points = 0.0
    for cat, cnt in counts.items():
        base = SEVERITY_POINTS.get(CATEGORY_SEVERITY.get(cat, "low"), 1)
        adj = base * cnt
        if cat == "help-seeking":
            adj *= HELP_SEEKING_DOWNWEIGHT
        total_points += adj

    distinct_cats = len(counts)

    # decision rules (conservative):
    # - HIGH only if:
    #     (a) >=2 distinct categories with at least one 'high' cat; OR
    #     (b) 'suicide methods' present AND immediacy cue.
    # - MODERATE if total_points >= 3 (multiple moderate hits or one strong + support).
    # - LOW if some evidence but below 3 points.
    # - NONE if no evidence or only help-seeking.
    if counts and not (set(counts.keys()) == {"help-seeking"}):
        if (distinct_cats >= 2 and high_present) or ("suicide methods" in counts and immediacy):
            severity = "high"
        elif total_points >= 3:
            severity = "moderate"
        else:
            severity = "low"
        # crisis type preference
        if "suicide methods" in counts:
            crisis_type = "self_harm"
        elif "suicidal thoughts" in counts:
            crisis_type = "suicidal_ideation"
        else:
            crisis_type = "other"
    else:
        crisis_type, severity = "none", "none"

    # Very conservative "imminent" upgrade:
    ACTION_CUES = re.compile(r"\b(will|going to|about to|plan to|planning to|tonight)\b", re.I)
    if severity == "high" and immediacy and "suicide methods" in counts and ACTION_CUES.search(text):
        severity = "imminent"

    # evidence list
    evidence = [{"category": cat, "fragment": frag} for cat, lst in filtered.items() for frag, _ in lst]

    return {
        "counts": counts,
        "distinct_categories": distinct_cats,
        "immediacy": immediacy,
        "negated_evidence": negated_evidence,
        "total_points": total_points,
        "crisis_type": crisis_type,
        "severity": severity,
        "evidence": evidence,
    }


In [ ]:
# Enforce JSON by prompt + cleanup

def llm_json(system_prompt: str, user_prompt: str, model_name: str = 'qwen-plus') -> dict:
    messages = [
        {'role':'system','content': system_prompt},
        {'role':'user','content': user_prompt + '\n\nReturn JSON only.'}
    ]
    raw = chat_completion_json(messages, model=model_name)
    # Basic repair for stray code fences / text
    txt = raw.strip().strip('`')
    try:
        return json.loads(txt)
    except json.JSONDecodeError:
        # Attempt to extract JSON substring
        m = re.search(r'\{[\s\S]*\}$', txt)
        if m:
            return json.loads(m.group(0))
        raise

In [ ]:
# Deterministic hard-rule filters
import re

RED_FLAGS = {
    'self_harm_methods': re.compile(r'(how to|method|steps?|technique).{0,40}(kill|harm|suicide|self-harm)', re.I),
    'med_dosage': re.compile(r'\b(\d+\s*(mg|milligrams))\b', re.I),
    'prescription_advice': re.compile(r'\b(prescribe|dosage|increase|reduce)\b.{0,15}\b(ssri|fluoxetine|sertraline|bupropion|benzodiazepine)\b', re.I),
}

def hard_rule_failures(text: str):
    return [k for k, rx in RED_FLAGS.items() if rx.search(text)]


In [ ]:
# Orchestrator: Generator -> Judges -> Policy Gate

def policy_gate(candidate: CandidateReply, crisis: CrisisVerdict, safety: SafetyVerdict, app: AppropriatenessVerdict):
    # Hard rules first (deterministic deny)
    hard_hits = hard_rule_failures(candidate.message)
    if hard_hits:
        return False, {'reason':'hard_rules', 'hits': hard_hits}

    # Safety judge verdict
    if not safety.passed:
        return False, {'reason':'safety_llm', 'violations': safety.violations}

    # If crisis is high/imminent, ensure the candidate includes resources
    needs_resources = crisis.severity in {'high','imminent'}
    if needs_resources and ('offers_resources_yes_no' not in candidate.tags):
        return False, {'reason':'missing_resources_for_severe_case'}

    # On-topic / appropriateness check
    if not app.on_topic:
        return False, {'reason':'off_topic', 'missing': app.missing_elements}

    return True, {}

def run_pipeline(user_text: str, policy_text: str, gen_model: str='qwen-plus', judge_model: str='qwen-plus'):
    # 1) Candidate generation
    gen_payload = f"{GENERATOR_JSON_SHAPE}\n\nUSER:\n{user_text}"
    cand_json = llm_json(GENERATOR_SYS, gen_payload, model_name=gen_model)
    candidate = CandidateReply(**cand_json)

    # 2) Judges
    crisis_payload = f"USER:\n{user_text}\n\nASSISTANT:\n{candidate.message}"
    crisis = CrisisVerdict(**llm_json(CRISIS_EVAL_SYS, crisis_payload, model_name=judge_model))

    safety_policy = 'Safety protocol (authoritative):\n' + policy_text + '\n\nEvaluate only the ASSISTANT reply below.'
    safety = SafetyVerdict(**llm_json(SAFETY_EVAL_SYS + '\n\n' + safety_policy, crisis_payload, model_name=judge_model))

    app = AppropriatenessVerdict(**llm_json(APPROPRIATENESS_SYS, crisis_payload, model_name=judge_model))

    # 3) Gate
    ok, meta = policy_gate(candidate, crisis, safety, app)
    if not ok:
        safe_stub = (
            "I’m here to support you. I can’t provide medical or medication advice. "
            "If you’re in immediate danger or thinking about harming yourself, please seek immediate help. "
            "In the U.S., you can call or text 988 for the Suicide & Crisis Lifeline. "
            "Would you like resources or to talk through what you’re feeling right now?"
        )
        return {'approved': False, 'reason': meta, 'message': safe_stub,
                'candidate': candidate.model_dump(), 'crisis': crisis.model_dump(),
                'safety': safety.model_dump(), 'appropriateness': app.model_dump()}

    return {'approved': True, 'message': candidate.message,
            'candidate': candidate.model_dump(), 'crisis': crisis.model_dump(),
            'safety': safety.model_dump(), 'appropriateness': app.model_dump()}